In [ ]:
# 필요한 라이브러리 import
from transformers import BitsAndBytesConfig  # 양자화 설정을 위한 transformers 모듈
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline  # HuggingFace 파이프라인 및 채팅 래퍼
import torch  # 디바이스(CPU/GPU) 확인용

# 4비트 양자화 설정
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

# 디바이스 설정: GPU가 있으면 0, 없으면 -1(CPU)
device = 0 if torch.cuda.is_available() else -1

# HuggingFace Hub에서 모델을 파이프라인 형태로 불러오기
# task를 'text-generation'으로 지정해서 텍스트 생성용 파이프라인을 만든다.
chat_model = HuggingFacePipeline.from_model_id(
    model_id="LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct",
    task="text-generation",
    pipeline_kwargs=dict(
      max_new_tokens=1024,  # 생성할 최대 토큰 수
      do_sample=False,     # 샘플링 비활성화(결정적 생성)
      repetition_penalty=1.03,  # 반복 패널티
    ),
    device=device  # 0 for first GPU, -1 for CPU
)

# 생성된 파이프라인을 ChatHuggingFace에 전달해서 채팅 형태로 사용할 수 있다.
llm = ChatHuggingFace(llm=chat_model)
